In [7]:
import re
import numpy as np
import pandas as pd
from collections import defaultdict
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
docs = {
 0: "Python is a popular programming language for data science and machine learning.",
 1: "Machine learning algorithms learn patterns from data to make predictions.",
 2: "The Indian cricket team won the world cup final in a thrilling match.",
 3: "Football world cup is watched by millions of fans around the world.",
 4: "Deep learning uses neural networks with many layers for image and speech tasks.",
 5: "Healthy diet and regular exercise keep the heart strong and healthy.",
 6: "Stock market investors analyse company profits and economic growth.",
 7: "Natural language processing allows computers to understand human language.",
 8: "Yoga and meditation improve mental health and reduce stress.",
 9: "The football team scored a late goal to win the league match.",
}
relevant = {"machine learning": {0, 1, 4}, "world cup": {2, 3}, "healthy life": {5, 8}}   # ground truth for evaluation

In [9]:
stem = PorterStemmer()
def preprocess(text):
    toks = re.findall(r"[a-z]+", text.lower())
    return [stem.stem(t) for t in toks if t not in ENGLISH_STOP_WORDS and len(t) > 1]

print(preprocess(docs[0]))

['python', 'popular', 'program', 'languag', 'data', 'scienc', 'machin', 'learn']


In [10]:
inverted = defaultdict(set)
for d, t in docs.items():
    for term in preprocess(t):
        inverted[term].add(d)
for term in ["learn", "world", "footbal"]:
    print(f"{term:8s} -> docs {sorted(inverted[term])}")

learn    -> docs [0, 1, 4]
world    -> docs [2, 3]
footbal  -> docs [3, 9]


In [11]:
class SearchEngine:
    def __init__(self, docs):
        self.ids = list(docs)
        self.docs = docs
        self.vec = TfidfVectorizer(tokenizer=preprocess, lowercase=False, token_pattern=None)
        self.matrix = self.vec.fit_transform([docs[i] for i in self.ids])

    def search(self, query, k=3):
        scores = cosine_similarity(self.vec.transform([query]), self.matrix)[0]
        order = scores.argsort()[::-1][:k]
        return [(self.ids[i], round(float(scores[i]), 3)) for i in order if scores[i] > 0]

engine = SearchEngine(docs)
for q in ["machine learning", "world cup", "healthy life", "language understanding"]:
    print(f"\nQuery: {q!r}")
    for d, s in engine.search(q):
        print(f"  doc {d} | score {s} | {docs[d]}")


Query: 'machine learning'
  doc 1 | score 0.585 | Machine learning algorithms learn patterns from data to make predictions.
  doc 0 | score 0.436 | Python is a popular programming language for data science and machine learning.
  doc 4 | score 0.167 | Deep learning uses neural networks with many layers for image and speech tasks.

Query: 'world cup'
  doc 3 | score 0.666 | Football world cup is watched by millions of fans around the world.
  doc 2 | score 0.428 | The Indian cricket team won the world cup final in a thrilling match.

Query: 'healthy life'
  doc 5 | score 0.667 | Healthy diet and regular exercise keep the heart strong and healthy.

Query: 'language understanding'
  doc 7 | score 0.625 | Natural language processing allows computers to understand human language.
  doc 0 | score 0.212 | Python is a popular programming language for data science and machine learning.


In [12]:
def evaluate(q, k=3):
    retrieved = [d for d, _ in engine.search(q, k)]
    hits = len(set(retrieved) & relevant[q])
    return hits / max(len(retrieved), 1), hits / len(relevant[q])

rows = []
for q in relevant:
    p, r = evaluate(q)
    rows.append((q, round(p, 2), round(r, 2), round(2*p*r/(p+r), 2) if p + r else 0))
res = pd.DataFrame(rows, columns=["query", "precision@3", "recall@3", "F1"])
display(res)
print("Mean F1:", res.F1.mean().round(3))

,query,precision@3,recall@3,F1
0,machine learning,1.0,1.0,1.00
1,world cup,1.0,1.0,1.00
2,healthy life,1.0,0.5,0.67


Mean F1: 0.89
